# ⚡ RAIZEN: Enterprise Full-Stack Coding Intelligence
### 🚀 High-Throughput vLLM PagedAttention GPU Streaming Engine & Cloudflare Tunnel

---

<p align="center">
  <b>Architected, Fine-Tuned & Created by <a href="https://shawaz.vercel.app/" target="_blank">SHAWAZ</a></b>
</p>

<p align="center">
  <a href="https://shawaz.vercel.app/" target="_blank"><img src="https://img.shields.io/badge/Creator-SHAWAZ-blue.svg?style=for-the-badge" alt="Creator"></a>
  <a href="https://shawaz.vercel.app/" target="_blank"><img src="https://img.shields.io/badge/Portfolio-shawaz.vercel.app-green.svg?style=for-the-badge" alt="Portfolio"></a>
  <a href="https://huggingface.co/shawaz03/RAIZEN" target="_blank"><img src="https://img.shields.io/badge/HuggingFace-shawaz03%2FRAIZEN-orange.svg?style=for-the-badge" alt="HuggingFace"></a>
  <img src="https://img.shields.io/badge/Model-RAIZEN--7B-red.svg?style=for-the-badge" alt="Model">
  <img src="https://img.shields.io/badge/Engine-vLLM_PagedAttention-purple.svg?style=for-the-badge" alt="Engine">
  <img src="https://img.shields.io/badge/Throughput-8--15_tok%2Fs-brightgreen.svg?style=for-the-badge" alt="Throughput">
  <img src="https://img.shields.io/badge/Hosting_Cost-%240.00%20(Free%20GPU)-gold.svg?style=for-the-badge" alt="Cost">
</p>

---

## 📖 Instructions
1. **Enable Free GPU**: Go to `Runtime` ➔ `Change runtime type` ➔ Select **T4 GPU** (or A100 / L4 if available).
2. **Start Backend Engine**: Click `Runtime` ➔ `Run all` (or press `Ctrl + F9`).
3. **Connect Frontend**: Copy the generated **Cloudflare Tunnel URL** (`https://*.trycloudflare.com`) and paste it into the **RAIZEN Chat Studio** frontend.

---

In [ ]:
# Cell 1: Environment & Dependency Installation
print("📦 [1/4] Installing RAIZEN vLLM core dependencies (vLLM, FastAPI, Uvicorn, Accelerate, BitsAndBytes)...")
!pip install -q -U vllm fastapi uvicorn pydantic accelerate bitsandbytes
print("✅ High-throughput vLLM engine dependencies installed successfully!")

In [ ]:
# Cell 2: Download & Configure Cloudflare Quick Tunnel
print("🌐 [2/4] Downloading & configuring Cloudflare Quick Tunnel binary...")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared --version
print("✅ Cloudflare Tunnel binary is configured and ready for zero-config public tunneling!")

In [ ]:
# Cell 3: Load RAIZEN Tokenizer
import os
import torch
from transformers import AutoTokenizer

MODEL_ID = "shawaz03/RAIZEN"

print(f"📥 [3/4] Initializing RAIZEN Tokenizer from Hugging Face ({MODEL_ID})...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side="right"
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Tokenizer ready! Vocab size: {len(tokenizer):,}, Pad token: '{tokenizer.pad_token}'")

In [ ]:
# Cell 4: Initialize High-Throughput vLLM PagedAttention Engine
import asyncio
from vllm import AsyncLLMEngine, AsyncEngineArgs, SamplingParams
from vllm.utils import random_uuid

print("🧠 [4/4] Initializing vLLM PagedAttention Engine with 4-bit quantization...")

engine_args = AsyncEngineArgs(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    dtype="bfloat16",
    max_model_len=4096,
    gpu_memory_utilization=0.90,
    trust_remote_code=True,
    enforce_eager=True,
)

engine = AsyncLLMEngine.from_engine_args(engine_args)

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
print(f"🎉 RAIZEN 7.61B vLLM Engine loaded successfully on {gpu_name}!")
print(f"📊 GPU VRAM Allocated: {vram_gb:.2f} GB (PagedAttention dynamic KV-cache ready: 8-15 tokens/sec)")

In [ ]:
# Cell 5: Warmup Inference & CUDA Kernel Pre-Compilation
import time

print("⚡ Pre-compiling vLLM CUDA kernels with warmup inference...")
warmup_messages = [
    {"role": "system", "content": "You are RAIZEN, an elite AI coding intelligence created by SHAWAZ (https://shawaz.vercel.app/)."},
    {"role": "user", "content": "Hello RAIZEN, confirm your operational readiness in one sentence."}
]

t0 = time.time()
warmup_text = tokenizer.apply_chat_template(warmup_messages, tokenize=False, add_generation_prompt=True)
warmup_params = SamplingParams(max_tokens=32, temperature=0.2)
warmup_request_id = f"warmup-{random_uuid()}"

async def run_warmup():
    results_generator = engine.generate(warmup_text, warmup_params, warmup_request_id)
    final_output = None
    async for request_output in results_generator:
        final_output = request_output
    return final_output.outputs[0].text if final_output else ""

try:
    loop = asyncio.get_event_loop()
except RuntimeError:
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

warmup_resp = loop.run_until_complete(run_warmup()).strip()
elapsed = time.time() - t0

print(f"✅ Warmup successful in {elapsed:.2f}s!")
print(f"🤖 RAIZEN Warmup Output: \"{warmup_resp}\"")

In [ ]:
# Cell 6: FastAPI Application & Cross-Origin Resource Sharing (CORS) Setup
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
import threading
import json

app = FastAPI(
    title="RAIZEN High-Throughput vLLM API",
    description="High-performance vLLM streaming inference engine architected by SHAWAZ (https://shawaz.vercel.app/)",
    version="1.0.0"
)

# Enable open CORS for Next.js frontend connection
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("✅ FastAPI application initialized with universal CORS middleware!")

In [ ]:
# Cell 7: Health Check & System Status Endpoints
import time

@app.get("/")
@app.get("/health")
async def health_check():
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    vram_gb = torch.cuda.memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0.0
    return {
        "status": "healthy",
        "model": "RAIZEN-7B",
        "version": "1.0.0",
        "creator": "SHAWAZ",
        "portfolio": "https://shawaz.vercel.app/",
        "huggingface": "https://huggingface.co/shawaz03/RAIZEN",
        "gpu": gpu_name,
        "vram_allocated_gb": round(vram_gb, 2),
        "precision": "vLLM PagedAttention (4-bit NF4)",
        "timestamp": int(time.time()),
    }

print("✅ Health check endpoints (/ and /health) configured!")

In [ ]:
# Cell 8: OpenAI-Compatible Streaming Chat Completions Endpoint (/v1/chat/completions)
class Message(BaseModel):
    role: str
    content: str

class ChatCompletionRequest(BaseModel):
    messages: List[Message]
    temperature: Optional[float] = 0.2
    top_p: Optional[float] = 0.95
    max_tokens: Optional[int] = Field(default=2048, le=4096)
    stream: Optional[bool] = True
    repetition_penalty: Optional[float] = 1.05

SYSTEM_DIRECTIVE = "You are RAIZEN, an elite AI coding intelligence created by SHAWAZ (https://shawaz.vercel.app/). You specialize in high-aesthetic UI/UX design systems, production React/Next.js/Tailwind components, resilient async backends, root-cause debugging, and optimized SQL."

@app.post("/v1/chat/completions")
async def chat_completions(req: ChatCompletionRequest):
    raw_messages = [{"role": m.role, "content": m.content} for m in req.messages]
    
    # Auto-inject RAIZEN identity system prompt if not present
    if not any(m["role"] == "system" for m in raw_messages):
        raw_messages.insert(0, {"role": "system", "content": SYSTEM_DIRECTIVE})
    
    try:
        prompt_text = tokenizer.apply_chat_template(
            raw_messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Chat template formatting error: {str(e)}")
    
    sampling_params = SamplingParams(
        temperature=req.temperature if req.temperature is not None else 0.2,
        top_p=req.top_p if req.top_p is not None else 0.95,
        max_tokens=req.max_tokens if req.max_tokens is not None else 2048,
        repetition_penalty=req.repetition_penalty if req.repetition_penalty is not None else 1.05,
    )
    
    req_id = f"chat-{random_uuid()}"
    
    if req.stream:
        async def generate_sse():
            created_ts = int(time.time())
            previous_text = ""
            results_generator = engine.generate(prompt_text, sampling_params, req_id)
            
            async for request_output in results_generator:
                current_text = request_output.outputs[0].text
                new_token = current_text[len(previous_text):]
                previous_text = current_text
                
                if new_token:
                    chunk = {
                        "id": f"chatcmpl-{created_ts}",
                        "object": "chat.completion.chunk",
                        "created": created_ts,
                        "model": "RAIZEN-7B",
                        "choices": [{
                            "index": 0,
                            "delta": {"content": new_token},
                            "finish_reason": None
                        }]
                    }
                    yield f"data: {json.dumps(chunk)}\n\n"
            
            yield "data: [DONE]\n\n"
        
        return StreamingResponse(
            generate_sse(),
            media_type="text/event-stream",
            headers={
                "Cache-Control": "no-cache",
                "Connection": "keep-alive",
                "X-Accel-Buffering": "no",
            }
        )
    else:
        results_generator = engine.generate(prompt_text, sampling_params, req_id)
        final_output = None
        async for request_output in results_generator:
            final_output = request_output
        
        response_text = final_output.outputs[0].text if final_output else ""
        return {
            "id": f"chatcmpl-{int(time.time())}",
            "object": "chat.completion",
            "created": int(time.time()),
            "model": "RAIZEN-7B",
            "choices": [{
                "index": 0,
                "message": {"role": "assistant", "content": response_text},
                "finish_reason": "stop"
            }]
        }

print("✅ High-throughput streaming endpoint (/v1/chat/completions) with vLLM PagedAttention & SSE configured!")

In [ ]:
# Cell 9: Start Background Uvicorn Server (Task 4.4.1)
import uvicorn
import threading
import time

def run_uvicorn():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_uvicorn, daemon=True)
server_thread.start()

# Brief pause to ensure port 8000 is listening
time.sleep(2)
print("🚀 [1/3] FastAPI Uvicorn server is actively listening in the background on http://0.0.0.0:8000!")

In [ ]:
# Cell 10: Launch Cloudflare Quick Tunnel (Task 4.4.2)
import subprocess

print("🌐 [2/3] Initializing Cloudflare Quick Tunnel on port 8000...")

# Spawn cloudflared tunnel process in background with stdout/stderr piped
tunnel_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

print("✅ Cloudflare Tunnel subprocess active and connecting to edge network!")

In [ ]:
# Cell 11: Extract & Display Public Cloudflare Tunnel URL (Task 4.4.3)
import re
import sys
from IPython.display import display, HTML

print("🔍 [3/3] Discovering assigned public Cloudflare HTTPS endpoint...")
public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

for _ in range(40):
    line = tunnel_process.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = url_pattern.search(line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    banner_html = f"""
    <div style="background: linear-gradient(135deg, #0f172a 0%, #1e1b4b 100%); padding: 24px; border-radius: 16px; border: 2px solid #6366f1; color: white; font-family: sans-serif; box-shadow: 0 10px 25px rgba(0,0,0,0.5);">
        <div style="display: flex; align-items: center; gap: 12px; margin-bottom: 12px;">
            <span style="font-size: 28px;">⚡</span>
            <h2 style="margin: 0; color: #818cf8; font-size: 22px;">RAIZEN 7.61B vLLM GPU ENGINE IS LIVE!</h2>
        </div>
        <p style="color: #94a3b8; margin: 4px 0 16px 0; font-size: 14px;">
            Architected & Created by <b style="color: #38bdf8;">SHAWAZ</b> | <a href="https://shawaz.vercel.app/" target="_blank" style="color: #4ade80; text-decoration: underline;">Portfolio: shawaz.vercel.app</a> | <b style="color: #a855f7;">vLLM PagedAttention Accelerated (8-15 tok/s)</b>
        </p>
        <div style="background: #020617; padding: 14px 18px; border-radius: 10px; border: 1px dashed #475569; margin: 16px 0;">
            <div style="font-size: 11px; text-transform: uppercase; color: #94a3b8; font-weight: bold; letter-spacing: 0.05em;">Your Public Cloudflare Tunnel URL:</div>
            <div style="font-family: monospace; font-size: 18px; color: #38bdf8; font-weight: bold; margin-top: 6px; word-break: break-all;">{public_url}</div>
        </div>
        <div style="font-size: 13px; color: #cbd5e1; line-height: 1.6;">
            <b>👉 How to connect:</b><br/>
            1. Copy the URL above.<br/>
            2. Open the <b>RAIZEN Chat Studio</b> frontend on your browser.<br/>
            3. Paste the URL into the <b>'Connect Backend'</b> dialog to begin real-time streaming & live code previews!
        </div>
    </div>
    """
    display(HTML(banner_html))
    print(f"\n🔗 Raw Public URL: {public_url}")
else:
    print("⚠️ Tunnel URL discovery timed out. Please check tunnel_process logs.", file=sys.stderr)


In [ ]:
# Cell 12: Automated Colab Keep-Alive Watchdog Loop (Task 4.4.4)
import urllib.request
import threading
import time

def keep_alive_watchdog():
    print("🛡️ [Watchdog] Keep-alive thread started (pinging /health every 3m)...")
    while True:
        time.sleep(180)  # Wait 3 minutes
        try:
            req = urllib.request.Request("http://localhost:8000/health", headers={"User-Agent": "RAIZEN-Watchdog"})
            with urllib.request.urlopen(req, timeout=5) as response:
                pass
        except Exception:
            pass

watchdog_thread = threading.Thread(target=keep_alive_watchdog, daemon=True)
watchdog_thread.start()

print("🛡️ RAIZEN Keep-Alive watchdog is running in the background!")
print("🎉 [vLLM ENGINE DEPLOYED] All Colab backend cells are operational!")